# Leave-One-List-Out Cross-Validation

> Fit per-subject on 8 lists, evaluate held-out NLL on the remaining list, across all 9 folds.

## Overview

Cross-validation provides a more rigorous model comparison than AIC by directly
measuring out-of-sample prediction. For each fold, parameters are optimized on 8
of 9 lists per subject, and the held-out list's negative log-likelihood is evaluated
under those fitted parameters. The aggregated held-out NLL across all 9 folds gives
one CV score per subject per model, which can be compared across models using paired
tests without requiring an explicit complexity penalty.

In [ ]:
import json
import os
import warnings
from pathlib import Path
from typing import Type

import numpy as np
from jaxcmr.cross_validation import cross_validate
from jaxcmr.helpers import (
    find_project_root,
    generate_trial_mask,
    import_from_string,
    load_data,
)
from jaxcmr.typing import FittingAlgorithm, LossFnGenerator

warnings.filterwarnings("ignore")

In [ ]:
# Data
data_tag = "TalmiEEG"
data_path = "data/TalmiEEG.h5"
trial_query = "data['subject'] > -1"
target_directory = "projects/TalmiEEG/results/"
max_subjects = 0

# Model
model_name = "EEGEmotionOnly"
make_factory_path = "jaxcmr.models_eeg.eeg_cmr.make_factory"
component_paths = {
    "mfc_create_fn": "jaxcmr.components.linear_memory.init_mfc",
    "mcf_create_fn": "jaxcmr.components.linear_memory.init_mcf",
    "context_create_fn": "jaxcmr.components.context.init",
    "termination_policy_create_fn": "jaxcmr.components.termination.NoStopTermination",
}
loss_fn_path = "jaxcmr.loss.set_permutation_likelihood.MemorySearchLikelihoodFnGenerator"
fit_alg_path = "jaxcmr.fitting.ScipyDE"
parameters = {
    "fixed": {
        "allow_repeated_recalls": False,
        "modulate_emotion_by_primacy": False,
        "learn_after_context_update": False,
        "lpp_main_scale": 0.0,
        "lpp_main_threshold": 0.0,
        "lpp_inter_scale": 0.0,
        "lpp_inter_threshold": 0.0,
    },
    "free": {
        "encoding_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
        "start_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
        "recall_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
        "shared_support": [2.220446049250313e-16, 100.0],
        "item_support": [2.220446049250313e-16, 100.0],
        "learning_rate": [2.220446049250313e-16, 0.9999999999999998],
        "primacy_scale": [2.220446049250313e-16, 100.0],
        "primacy_decay": [2.220446049250313e-16, 100.0],
        "choice_sensitivity": [2.220446049250313e-16, 100.0],
        "emotion_scale": [2.220446049250313e-16, 10.0],
    },
}

# CV settings
fold_field = "list"
cv_best_of = 1
base_run_tag = "50_set_likelihood_fixed_term"
best_of = 3

# DE hyperparameters
relative_tolerance = 0.001
popsize = 15
num_steps = 1000
cross_rate = 0.9
diff_w = 0.85

# Flow
redo_cv = True

In [ ]:
run_tag = f"{base_run_tag}_best_of_{best_of}"
if max_subjects:
    run_tag += f"_nsubs_{max_subjects}"

product_dir = os.path.join(target_directory, "fits")
os.makedirs(product_dir, exist_ok=True)

project_root = Path(find_project_root())
data = load_data(os.path.join(project_root, data_path), max_subjects)
trial_mask = generate_trial_mask(data, trial_query)

make_factory = import_from_string(make_factory_path)
model_factory = make_factory(
    **{key: import_from_string(path) for key, path in component_paths.items()}
)

fitting_algorithm: Type[FittingAlgorithm] = import_from_string(fit_alg_path)
loss_fn_generator: Type[LossFnGenerator] = import_from_string(loss_fn_path)

fitter = fitting_algorithm(
    data,
    None,
    parameters["fixed"],
    model_factory,
    loss_fn_generator,
    hyperparams={
        "num_steps": num_steps,
        "pop_size": popsize,
        "relative_tolerance": relative_tolerance,
        "cross_over_rate": cross_rate,
        "diff_w": diff_w,
        "progress_bar": True,
        "display_iterations": False,
        "best_of": best_of,
        "bounds": parameters["free"],
    },
)

In [ ]:
cv_path = Path(product_dir) / f"{data_tag}_{model_name}_{run_tag}_cv.json"
metadata = {
    "run_tag": run_tag,
    "data_tag": data_tag,
    "data_query": trial_query,
    "model": model_name,
    "name": f"{data_tag}_{model_name}_{run_tag}",
    "components": component_paths,
    "fit_algorithm": fit_alg_path,
    "loss_function": loss_fn_path,
    "model_factory": make_factory_path,
    "cv_best_of": cv_best_of,
    "fold_field": fold_field,
}

if cv_path.exists() and not redo_cv:
    with cv_path.open() as handle:
        cv_result = json.load(handle)
    cv_result |= metadata
    print(f"Loaded from {cv_path}")
else:
    cv_result = cross_validate(
        fitter, trial_mask, fold_field=fold_field, best_of=cv_best_of
    )
    cv_result |= metadata
    with cv_path.open("w") as handle:
        json.dump(cv_result, handle, indent=4)
    print(f"Saved to {cv_path}")

In [ ]:
cv_fitness = np.array(cv_result["cv_fitness"])
print(f"Model: {model_name}")
print(f"Folds: {cv_result['n_folds']}")
print(f"Subjects: {len(cv_result['subjects'])}")
print(f"CV time: {cv_result['cv_time']:.0f}s")
print(f"Mean CV-NLL: {np.mean(cv_fitness):.2f} +/- {np.std(cv_fitness) / np.sqrt(len(cv_fitness)) * 1.96:.2f}")
print(f"Per-fold mean test NLL: {[f'{np.mean(f):.2f}' for f in cv_result['test_fitness']]}")